# Claude Conversations Export: EDA & Correlation Analysis

This notebook loads the 14 MB Claude JSON export, validates its structure, flattens the nested conversation/message data into analysis-friendly tables, and measures key correlations across conversation size, participation, and tool usage.

**Scope**
1. Load and validate the JSON schema
2. Summarize record counts, coverage, and missingness
3. Flatten into conversation-level and message-level tables
4. Run correlation analysis on conversation length, message counts, timing, and artifact usage
5. Create compact visualizations of the strongest relationships

## How I Used This Notebook to Get the Data

This flowchart shows the path I followed in the notebook to inspect the export, flatten the structure, and turn it into metrics for the dashboard.

```mermaid
flowchart TD
    A[Claude conversations export JSON] --> B[Load file from claude/data-*/conversations.json]
    B --> C[Validate top-level schema]
    C --> D[Inspect conversation and message structure]
    D --> E[Flatten nested data into conv_df and msg_df]
    E --> F[Analyze prompts, filler words, and constraints]
    F --> G[Build summary tables and correlations]
    G --> H[Export prompt_evaluation_metrics.csv]
    H --> I[Use metrics to generate dashboard]
```


In [23]:
import json
import os
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from collections import Counter, defaultdict
from typing import Any, Dict, List, Tuple

# Set notebook defaults
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)
pd.set_option('display.width', None)

# Locate the conversations.json file
base_path = Path('/Users/lohith/Documents/Data')
claude_dir = base_path / 'claude'
export_dirs = list(claude_dir.glob('data-*-batch-0000'))

if not export_dirs:
    print(f"No export directory found in {claude_dir}")
else:
    export_dir = export_dirs[0]
    json_file = export_dir / 'conversations.json'
    print(f"✓ Found export directory: {export_dir.name}")
    print(f"✓ Target file: {json_file.name}")
    print(f"✓ File size: {json_file.stat().st_size / 1e6:.1f} MB")

✓ Found export directory: data-f4ec4120-10da-47ee-8585-a6e7c33f8028-1778845878-17396ecd-batch-0000
✓ Target file: conversations.json
✓ File size: 13.7 MB


In [24]:
# Load the JSON
try:
    with open(json_file, 'r', encoding='utf-8') as f:
        conversations = json.load(f)
    print(f"✓ Loaded {len(conversations)} conversations")
except Exception as e:
    print(f"✗ Error loading JSON: {e}")
    conversations = []

# Inspect the top-level schema
print("\n--- Top-level structure ---")
if conversations:
    sample_conv = conversations[0]
    print(f"Type: {type(conversations)}")
    print(f"Sample keys: {list(sample_conv.keys())}")
    print(f"\nSample conversation (first 500 chars):")
    print(json.dumps(sample_conv, indent=2, ensure_ascii=False)[:500])

✓ Loaded 145 conversations

--- Top-level structure ---
Type: <class 'list'>
Sample keys: ['uuid', 'name', 'summary', 'created_at', 'updated_at', 'account', 'chat_messages']

Sample conversation (first 500 chars):
{
  "uuid": "ad8ada45-d306-4e2d-84ed-2a06d6bd8abc",
  "name": "",
  "summary": "",
  "created_at": "2025-08-13T16:10:51.609648Z",
  "updated_at": "2025-08-13T16:10:51.609648Z",
  "account": {
    "uuid": "719bf43f-395b-4556-907e-9591baf1352a"
  },
  "chat_messages": []
}


In [25]:
# Inspect message-level structure
print("\n--- Message-level structure ---")
if conversations and 'chat_messages' in conversations[0]:
    msgs = conversations[0]['chat_messages']
    if msgs:
        sample_msg = msgs[0]
        print(f"Messages in first conversation: {len(msgs)}")
        print(f"Sample message keys: {list(sample_msg.keys())}")
        print(f"\nSample message (first 400 chars):")
        print(json.dumps(sample_msg, indent=2, ensure_ascii=False)[:400])
else:
    print("No chat_messages found in sample conversation")


--- Message-level structure ---


In [26]:
# Check schema coverage and field types
print("\n--- Schema coverage ---")
required_fields = ['uuid', 'chat_messages', 'created_at', 'updated_at']
optional_fields = ['name', 'summary', 'account']

coverage = {}
for field in required_fields + optional_fields:
    present = sum(1 for c in conversations if field in c)
    coverage[field] = (present, len(conversations))

for field, (present, total) in coverage.items():
    pct = (present / total * 100) if total > 0 else 0
    status = "✓" if pct == 100 else "◐" if pct > 50 else "✗"
    print(f"{status} {field:20} {present:5} / {total:5} ({pct:5.1f}%)")

# Identify null/empty chat_messages
empty_conv = sum(1 for c in conversations if not c.get('chat_messages'))
print(f"\n✗ Conversations with empty/no chat_messages: {empty_conv}")


--- Schema coverage ---
✓ uuid                   145 /   145 (100.0%)
✓ chat_messages          145 /   145 (100.0%)
✓ created_at             145 /   145 (100.0%)
✓ updated_at             145 /   145 (100.0%)
✓ name                   145 /   145 (100.0%)
✓ summary                145 /   145 (100.0%)
✓ account                145 /   145 (100.0%)

✗ Conversations with empty/no chat_messages: 1


In [27]:
def flatten_conversations(conversations: List[Dict]) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Flatten nested conversation/message structure into two tables:
    - conv_table: one row per conversation with aggregated stats
    - msg_table: one row per message with parent conversation reference
    """
    conv_rows = []
    msg_rows = []
    
    for conv_idx, conv in enumerate(conversations):
        conv_uuid = conv.get('uuid', f'missing_{conv_idx}')
        conv_created = conv.get('created_at')
        conv_updated = conv.get('updated_at')
        conv_name = conv.get('name', '')
        conv_summary = conv.get('summary', '')
        account_uuid = conv.get('account', {}).get('uuid', '')
        
        messages = conv.get('chat_messages', [])
        
        # Conversation-level row
        if conv_created and conv_updated:
            try:
                created_dt = datetime.fromisoformat(conv_created.replace('Z', '+00:00'))
                updated_dt = datetime.fromisoformat(conv_updated.replace('Z', '+00:00'))
                duration_hours = (updated_dt - created_dt).total_seconds() / 3600
            except:
                created_dt, updated_dt, duration_hours = None, None, None
        else:
            created_dt, updated_dt, duration_hours = None, None, None
        
        # Count messages by sender
        human_msgs = sum(1 for m in messages if m.get('sender') == 'human')
        assistant_msgs = sum(1 for m in messages if m.get('sender') == 'assistant')
        
        # Look for tool usage in message content
        tool_count = 0
        artifact_count = 0
        for msg in messages:
            content = msg.get('content', [])
            if not isinstance(content, list):
                content = [content] if content else []
            for block in content:
                if isinstance(block, dict):
                    if block.get('type') == 'tool_use':
                        tool_count += 1
                    if block.get('type') == 'text' and 'tool' in block:
                        artifact_count += 1
        
        conv_rows.append({
            'conv_uuid': conv_uuid,
            'account_uuid': account_uuid,
            'name': conv_name,
            'summary': conv_summary,
            'created_at': created_dt,
            'updated_at': updated_dt,
            'duration_hours': duration_hours,
            'message_count': len(messages),
            'human_messages': human_msgs,
            'assistant_messages': assistant_msgs,
            'tool_uses': tool_count,
            'artifact_like': artifact_count,
        })
        
        # Message-level rows
        for msg_idx, msg in enumerate(messages):
            msg_uuid = msg.get('uuid', f'missing_{msg_idx}')
            msg_sender = msg.get('sender', '')
            msg_created = msg.get('created_at')
            msg_text = msg.get('text', '')
            
            # Try to parse message timestamps
            if msg_created:
                try:
                    msg_dt = datetime.fromisoformat(msg_created.replace('Z', '+00:00'))
                except:
                    msg_dt = None
            else:
                msg_dt = None
            
            msg_content = msg.get('content', [])
            if not isinstance(msg_content, list):
                msg_content = [msg_content] if msg_content else []
            
            content_types = [block.get('type', 'unknown') if isinstance(block, dict) else type(block).__name__ for block in msg_content]
            content_blocks = len(msg_content)
            
            msg_rows.append({
                'conv_uuid': conv_uuid,
                'msg_uuid': msg_uuid,
                'sender': msg_sender,
                'created_at': msg_dt,
                'text_length': len(msg_text),
                'content_blocks': content_blocks,
                'content_types': ', '.join(content_types),
                'parent_message_uuid': msg.get('parent_message_uuid'),
            })
    
    conv_df = pd.DataFrame(conv_rows)
    msg_df = pd.DataFrame(msg_rows)
    
    return conv_df, msg_df

# Run flattening
conv_df, msg_df = flatten_conversations(conversations)

print(f"✓ Flattened {len(conv_df)} conversations → {len(msg_df)} messages")
print(f"\nConversation table shape: {conv_df.shape}")
print(f"Message table shape: {msg_df.shape}")

✓ Flattened 145 conversations → 2217 messages

Conversation table shape: (145, 12)
Message table shape: (2217, 8)


In [34]:
# Select a specific conversation by its index. 
# Change this number to view a different chat (e.g., 1, 2, 3...)
target_idx = 55

# Ensure the index is within range
if target_idx < len(conversations):
    selected_chat = conversations[target_idx]
    
    # Extract metadata for context
    chat_name = selected_chat.get('name', 'Unnamed Chat')
    chat_date = selected_chat.get('created_at', 'Unknown Date')
    messages = selected_chat.get('chat_messages', [])
    
    print(f"=== CHAT LOG: {chat_name} ===")
    print(f"=== STARTED: {chat_date} ===\n")
    
    if not messages:
        print("No messages found in this conversation.")
    else:
        # Loop through the sequence of messages
        for msg in messages:
            # Capitalize the sender for cleaner formatting (Human -> Human, assistant -> Assistant)
            sender = str(msg.get('sender', 'Unknown')).capitalize()
            
            # Extract the actual text content
            text_content = msg.get('text', '').strip()
            
            # Format the output based on who is speaking
            if sender == 'Human':
                print(f"👤 {sender.upper()}:")
                print(f"{text_content}\n")
                print("-" * 50 + "\n")
                
            elif sender == 'Assistant':
                print(f"🤖 {sender.upper()}:")
                print(f"{text_content}\n")
                print("=" * 50 + "\n")
else:
    print(f"Error: Conversation index {target_idx} is out of bounds. You only have {len(conversations)} conversations.")

=== CHAT LOG: Portfolio website and Google AI tools ===
=== STARTED: 2026-02-14T17:38:44.167076Z ===

👤 HUMAN:
GIve me draft to post on linkedin about my new portfolio website, how i am enjoying using Google's suite of AI tools, mainly antigravity ,stitch and flow.

--------------------------------------------------

🤖 ASSISTANT:
I'll help you draft a LinkedIn post about your portfolio website and experience with Google's AI tools. However, I need to verify the names of these tools first, as I want to make sure I'm referencing them correctly.
```
This block is not supported on your current device yet.
```


```
This block is not supported on your current device yet.
```


```
This block is not supported on your current device yet.
```

Perfect! Now I have information about all three Google AI tools you mentioned. Let me create a LinkedIn post for you:

---

🚀 Excited to share my new portfolio website!

I've been exploring the incredible suite of AI tools Google has built, and I have to

In [35]:
import re

# Select a specific conversation by its index
target_idx = 1 

if target_idx < len(conversations):
    selected_chat = conversations[target_idx]
    
    chat_name = selected_chat.get('name', 'Unnamed Chat')
    messages = selected_chat.get('chat_messages', [])
    
    print(f"=== PROMPT EVALUATION REPORT: {chat_name} ===\n")
    
    # 1. Define our search patterns (You can customize these lists)
    # Looking for conversational fluff
    filler_pattern = r'\b(please|could you|if you don\'t mind|thanks|thank you|maybe|i think|just|wondering)\b'
    # Looking for strong directive constraints
    constraint_pattern = r'\b(must|do not|only|strictly|always|never|require|specifically|using)\b'
    
    human_turns = 0
    total_human_words = 0
    total_fillers = 0
    
    # 2. Iterate through the chat sequentially
    for msg in messages:
        sender = str(msg.get('sender', 'Unknown')).capitalize()
        text_content = msg.get('text', '').strip()
        
        if not text_content:
            continue
            
        word_count = len(text_content.split())
        
        if sender == 'Human':
            human_turns += 1
            total_human_words += word_count
            
            # Analyze the prompt text (converted to lowercase for matching)
            text_lower = text_content.lower()
            fillers_found = re.findall(filler_pattern, text_lower)
            constraints_found = re.findall(constraint_pattern, text_lower)
            
            total_fillers += len(fillers_found)
            
            print(f"👤 HUMAN PROMPT [{human_turns}]:")
            print(f"   - Length: {word_count} words")
            print(f"   - Filler words used: {len(fillers_found)} {list(set(fillers_found)) if fillers_found else ''}")
            print(f"   - Explicit constraints: {len(constraints_found)} {list(set(constraints_found)) if constraints_found else ''}")
            
            # Print a snippet of the prompt to remind you what it was
            snippet = text_content[:150].replace('\n', ' ') + '...' if len(text_content) > 150 else text_content
            print(f"   - Preview: \"{snippet}\"\n")
            
        elif sender == 'Assistant':
            # Check if Claude had to trigger hidden tools (like the block errors in your example)
            tool_blocks = text_content.count("This block is not supported")
            
            print(f"🤖 ASSISTANT RESPONSE [{human_turns}]:")
            print(f"   - Length: {word_count} words")
            if tool_blocks > 0:
                print(f"   - ⚠️ Tools/Search triggered: {tool_blocks} times before answering")
            print("-" * 60 + "\n")

    # 3. Print the final evaluation summary for this chat
    print("=== SUMMARY METRICS ===")
    print(f"Total Back-and-Forth Turns: {human_turns}")
    if human_turns > 0:
        print(f"Average Prompt Length: {total_human_words // human_turns} words")
        print(f"Overall Filler Density: {total_fillers} total filler words used")
        
        # Basic diagnostic logic
        if human_turns > 3:
            print("💡 INSIGHT: This took multiple turns. Review your first prompt to see if you missed critical context.")
        if total_fillers > (human_turns * 2):
            print("💡 INSIGHT: High filler word usage. Try being more direct to save tokens and improve clarity.")

else:
    print(f"Error: Conversation index {target_idx} is out of bounds.")

=== PROMPT EVALUATION REPORT: n8n Google Sheets to Email Workflow ===

👤 HUMAN PROMPT [1]:
   - Length: 15 words
   - Filler words used: 0 
   - Explicit constraints: 0 
   - Preview: "how do i do this on n8n: Create a **simple workflow** (Google Sheets → Email)"

🤖 ASSISTANT RESPONSE [1]:
   - Length: 395 words
------------------------------------------------------------

=== SUMMARY METRICS ===
Total Back-and-Forth Turns: 1
Average Prompt Length: 15 words
Overall Filler Density: 0 total filler words used


In [36]:
import re
from collections import Counter

# 1. Define search patterns
filler_pattern = r'\b(please|could you|if you don\'t mind|thanks|thank you|maybe|i think|just|wondering)\b'
constraint_pattern = r'\b(must|do not|only|strictly|always|never|require|specifically|using)\b'

# 2. Set up global tracking variables
total_conversations = len(conversations)
active_conversations = 0
total_human_turns = 0
total_human_words = 0
total_fillers = 0
total_constraints = 0

# Counter to find your most frequently used filler words
filler_counter = Counter()

# 3. Iterate through all conversations in the file
for conv in conversations:
    messages = conv.get('chat_messages', [])
    
    # Skip empty/ghost conversations
    if not messages:
        continue
        
    active_conversations += 1
    
    # Analyze messages within the active conversation
    for msg in messages:
        sender = str(msg.get('sender', 'Unknown')).capitalize()
        text_content = msg.get('text', '').strip()
        
        if not text_content:
            continue
            
        if sender == 'Human':
            total_human_turns += 1
            word_count = len(text_content.split())
            total_human_words += word_count
            
            # Regex analysis (lowercase to match patterns)
            text_lower = text_content.lower()
            fillers_found = re.findall(filler_pattern, text_lower)
            constraints_found = re.findall(constraint_pattern, text_lower)
            
            total_fillers += len(fillers_found)
            total_constraints += len(constraints_found)
            
            # Log specific filler words used
            filler_counter.update(fillers_found)

# 4. Calculate global averages safely
avg_words_per_prompt = (total_human_words // total_human_turns) if total_human_turns > 0 else 0
avg_turns_per_chat = (total_human_turns / active_conversations) if active_conversations > 0 else 0

# 5. Print the Global Summary
print("==================================================")
print("       GLOBAL PROMPT EVALUATION SUMMARY")
print("==================================================\n")

print(f"Total Chats Analyzed: {active_conversations} (Out of {total_conversations} total files)")
print(f"Total Prompts Sent:   {total_human_turns}")
print(f"Total Words Written:  {total_human_words}\n")

print("--- AVERAGE BEHAVIORS ---")
print(f"Average Prompts per Chat: {avg_turns_per_chat:.1f} turns")
print(f"Average Prompt Length:    {avg_words_per_prompt} words\n")

print("--- HABIT TRACKING ---")
print(f"Total Filler Words Used:  {total_fillers}")
print(f"Total Constraints Used:   {total_constraints}")

# Display the top 3 most used filler words if any exist
if total_fillers > 0:
    top_fillers = filler_counter.most_common(3)
    print("\nMost Common Filler Words:")
    for word, count in top_fillers:
        print(f"  - '{word}': {count} times")
        
print("\n==================================================")

       GLOBAL PROMPT EVALUATION SUMMARY

Total Chats Analyzed: 144 (Out of 145 total files)
Total Prompts Sent:   1094
Total Words Written:  52759

--- AVERAGE BEHAVIORS ---
Average Prompts per Chat: 7.6 turns
Average Prompt Length:    48 words

--- HABIT TRACKING ---
Total Filler Words Used:  173
Total Constraints Used:   218

Most Common Filler Words:
  - 'just': 75 times
  - 'please': 37 times
  - 'maybe': 29 times



In [37]:
import pandas as pd
import re

# 1. Define our search patterns
filler_pattern = r'\b(please|could you|if you don\'t mind|thanks|thank you|maybe|i think|just|wondering)\b'
constraint_pattern = r'\b(must|do not|only|strictly|always|never|require|specifically|using)\b'

# List to hold the row data for the CSV
csv_rows = []

# 2. Iterate through all conversations
for target_id, conv in enumerate(conversations):
    name = conv.get('name', f'Unnamed_Chat_{target_id}')
    messages = conv.get('chat_messages', [])
    
    human_turns = 0
    total_human_words = 0
    total_fillers = 0
    total_constraints = 0
    first_prompt_preview = "N/A"
    
    # Process each message in the conversation
    for msg in messages:
        sender = str(msg.get('sender', '')).capitalize()
        text_content = msg.get('text', '').strip()
        
        if not text_content:
            continue
            
        if sender == 'Human':
            # Capture a preview of the very first prompt in this chat
            if human_turns == 0:
                clean_text = text_content.replace('\n', ' ')
                first_prompt_preview = clean_text[:75] + '...' if len(clean_text) > 75 else clean_text

            human_turns += 1
            total_human_words += len(text_content.split())
            
            # Analyze text for fillers and constraints
            text_lower = text_content.lower()
            total_fillers += len(re.findall(filler_pattern, text_lower))
            total_constraints += len(re.findall(constraint_pattern, text_lower))
    
    # 3. Only append rows for chats that actually had human messages
    if human_turns > 0:
        avg_prompt_length = total_human_words // human_turns
        
        csv_rows.append({
            'Target_ID': target_id,
            'Chat_Name': name,
            'Total_Turns': human_turns,
            'Avg_Prompt_Length': avg_prompt_length,
            'Filler_Word_Count': total_fillers,
            'Explicit_Constraints': total_constraints,
            'First_Prompt_Preview': first_prompt_preview
        })

# 4. Convert to a Pandas DataFrame and save to CSV
df = pd.DataFrame(csv_rows)
output_filename = "prompt_evaluation_metrics.csv"
df.to_csv(output_filename, index=False)

print(f"✓ Successfully processed {len(conversations)} total entries.")
print(f"✓ Exported {len(df)} active chats to '{output_filename}'.")
print("\n--- First 5 Rows Preview ---")
display(df.head())

✓ Successfully processed 145 total entries.
✓ Exported 142 active chats to 'prompt_evaluation_metrics.csv'.

--- First 5 Rows Preview ---


,Target_ID,Chat_Name,Total_Turns,Avg_Prompt_Length,Filler_Word_Count,Explicit_Constraints,First_Prompt_Preview
0,1,n8n Google Sheets to Email Workflow,1,15,0,0,how do i do this on n8n: Create a **simple wor...
1,2,Coin Flip Streak Probability Simulation,2,271,0,2,"question: For this exercise, we’ll try doing a..."
2,3,AI Prompt Optimization Framework,16,60,11,8,"PROMPT: You are Lyra, a master-level AI prompt..."
3,4,Learning GNU Radio software benefits,3,174,0,4,what can i get out of learning this gnuradio s...
4,5,Building plot accuracy,4,19,1,0,start building plotsnr accuracy
